In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Import pour calcul des surfaces de cellules
from seapopym.transport import compute_spherical_cell_areas
from seapopym.standard.coordinates import Coordinates

In [ ]:
interval = "1W"


In [ ]:
lmtl_biomass = xr.open_mfdataset("./LMTL_Synthetic/output/*biomass*.nc")
lmtl_biomass = lmtl_biomass.biomass
lmtl_biomass = lmtl_biomass.rename({"longitude": "x", "latitude": "y"})
lmtl_biomass.time

In [ ]:
seapopym_biomass = xr.open_dataset("./SeapoPym_Synthetic.zarr")
seapopym_biomass = seapopym_biomass.biomass
seapopym_biomass

In [ ]:
dt = 3
n = 1e10
pos = 0
for i in range(10):
    val = float((lmtl_biomass.isel(time=0) - seapopym_biomass.isel(time=i)).max())
    print("i = ", i, " : ", val)
    if val < n:
        pos = i
        n = val

seapopym_biomass = seapopym_biomass.isel(time=slice(pos, None, int(24 / dt)))
seapopym_biomass

In [ ]:
lmtl_biomass = lmtl_biomass.assign_coords({"time": lmtl_biomass.time.dt.floor("D")})
seapopym_biomass = seapopym_biomass.assign_coords({"time": seapopym_biomass.time.dt.floor("D")})

In [ ]:
lmtl_biomass.load()
_ = seapopym_biomass.load()

# Calculer les surfaces des cellules (nécessaire pour les métriques pondérées)
lat = lmtl_biomass[Coordinates.Y.value]
lon = lmtl_biomass[Coordinates.X.value]
cell_areas = compute_spherical_cell_areas(lat, lon)
total_area = cell_areas.sum()

print(f"✅ Cell areas computed: {cell_areas.shape}")
print(f"Total domain area: {total_area.values / 1e12:.2f} × 10^12 m²")

In [ ]:
## MAPE ------------------------------------------------------------------------------------------------------------- ##
mape = np.abs((lmtl_biomass - seapopym_biomass) / (lmtl_biomass + 1e-10)) * 100
mape.attrs = {"units": "%"}
mpe = ((lmtl_biomass - seapopym_biomass) / (lmtl_biomass + 1e-10)) * 100
mpe.attrs = {"units": "%"}
max = 50

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(18, 6))
mape.isel(time=slice(None, None)).mean("time").plot(vmax=50, cmap="viridis", ax=axis[0])
axis[0].set_title("MAPE")
mpe.isel(time=slice(None, None)).mean("time").plot(vmax=50, cmap="RdYlGn_r", ax=axis[1])
axis[1].set_title("MPE")

plt.show()

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(9, 3))

# ✅ CORRECTED: Conservation de masse avec pondération par surface
seapopym_total_mass = (seapopym_biomass * cell_areas).sum(["x", "y"])
lmtl_total_mass = (lmtl_biomass * cell_areas).sum(["x", "y"])

seapopym_growth_rate = (seapopym_total_mass / seapopym_total_mass.isel(time=0) * 100) - 100
seapopym_growth_rate.plot(ax=axis[0], label="Seapopym")

lmtl_growth_rate = (lmtl_total_mass / lmtl_total_mass.isel(time=0) * 100) - 100
lmtl_growth_rate.plot(ax=axis[0], label="LMTL")

axis[0].set_title("Total Mass Conservation (area-weighted)")
axis[0].set_xlabel("Time")
axis[0].set_ylabel("Variation (%)")
axis[0].grid()
axis[0].legend()

# ✅ CORRECTED: Moyenne spatiale de MAPE pondérée par surface
mape_spatial_mean = (mape * cell_areas).sum(["x", "y"]) / total_area
mape_spatial_mean.plot(ax=axis[1])
axis[1].set_title("MAPE (area-weighted spatial mean)")
axis[1].grid()
axis[1].set_xlabel("Time")
axis[1].set_ylabel("MAPE (%)")
axis[1].set_ylim(0, None)

plt.tight_layout()
plt.show()

In [ ]:
# Calculer les moyennes hebdomadaires
mape_weekly = mape.resample(time=interval).mean()
mpe_weekly = mpe.resample(time=interval).mean()
lmtl_weekly = lmtl_biomass.resample(time=interval).mean()
seapopym_weekly = seapopym_biomass.resample(time=interval).mean()

seapopym_weekly = xr.where(seapopym_weekly.notnull(), seapopym_weekly, 0)
lmtl_weekly = xr.where(lmtl_weekly.notnull(), lmtl_weekly, 0)

# Créer la figure pour l'animation
fig = plt.figure(figsize=(18, 12))


def update(frame):
    """Fonction de mise à jour pour chaque frame de l'animation"""
    # Effacer complètement la figure
    fig.clear()

    # Recréer les subplots en grille 2x2
    ax1 = fig.add_subplot(2, 2, 1)
    ax2 = fig.add_subplot(2, 2, 2)
    ax3 = fig.add_subplot(2, 2, 3)
    ax4 = fig.add_subplot(2, 2, 4)

    # MAPE (haut gauche)
    mape_weekly.isel(time=frame).plot(vmax=max, cmap="viridis", ax=ax1)
    ax1.set_title(f"MAPE - Semaine {frame + 1}")

    # MPE (haut droite)
    mpe_weekly.isel(time=frame).plot(vmax=max, vmin=-max, cmap="RdYlGn_r", ax=ax2)
    ax2.set_title(f"MPE - Semaine {frame + 1}")

    # Biomasse LMTL (bas gauche)
    lmtl_weekly.isel(time=frame).plot(cmap="viridis", ax=ax3, vmax=lmtl_weekly.max() / 100)
    ax3.set_title(f"Biomasse LMTL - Semaine {frame + 1}")

    # Biomasse SeapoPym (bas droite)
    seapopym_weekly.isel(time=frame).plot(cmap="viridis", ax=ax4, vmax=seapopym_weekly.max() / 100)
    ax4.set_title(f"Biomasse SeapoPym - Semaine {frame + 1}")

    plt.tight_layout()


# Créer l'animation
n_frames = len(mape_weekly.time)
anim = FuncAnimation(fig, update, frames=n_frames, interval=500, repeat=True)

# Pour sauvegarder l'animation (décommenter si nécessaire)
_ = anim.save("mape_mpe_animation.gif", writer="pillow", fps=10)

# RMSE

In [ ]:
## RMSE ------------------------------------------------------------------------------------------------------------- ##
# ✅ CORRECTED: RMSE pondéré par surface des cellules

# RMSE pondéré par surface (métrique globale temporelle)
squared_error = (lmtl_biomass - seapopym_biomass) ** 2
weighted_squared_error = squared_error * cell_areas
rmse_global = np.sqrt(weighted_squared_error.sum(["x", "y"]) / total_area)
rmse_global.attrs = {"units": "g/m²"}

# RMSE spatial (2D) pour visualisation
rmse_spatial = np.sqrt(squared_error.mean("time"))
rmse_spatial.attrs = {"units": "g/m²"}

# NRMSE avec normalisation pondérée
weighted_biomass = lmtl_biomass * cell_areas
mean_weighted = weighted_biomass.sum(["x", "y"]) / total_area
variance_weighted = (((lmtl_biomass - mean_weighted) ** 2) * cell_areas).sum(
    ["x", "y"]
) / total_area
std_weighted = np.sqrt(variance_weighted)

nrmse_global = rmse_global / std_weighted
nrmse_global.attrs = {"units": "dimensionless"}

# NRMSE spatial pour visualisation
nrmse_spatial = rmse_spatial / lmtl_biomass.std(dim="time")
nrmse_spatial.attrs = {"units": "dimensionless"}

vmin = 0
vmax_rmse = float(rmse_spatial.quantile(0.95))  # Use 95th percentile for better visualization
vmax_nrmse = 1.0

print(f"Global RMSE (time-averaged): {rmse_global.mean().values:.4e} g/m²")
print(f"Global NRMSE (time-averaged): {nrmse_global.mean().values:.4f}")

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(18, 6))

# Plot spatial RMSE (averaged over time)
rmse_spatial.plot(cmap="viridis", ax=axis[0], vmin=vmin, vmax=vmax_rmse)
axis[0].set_title("RMSE (temporal mean)")

# Plot spatial NRMSE (averaged over time)
nrmse_spatial.plot(cmap="RdYlGn_r", ax=axis[1], vmin=vmin, vmax=vmax_nrmse)
axis[1].set_title("NRMSE (temporal mean)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(9, 3))

# ✅ Mass conservation (already computed with area-weighting)
seapopym_growth_rate.plot(ax=axis[0], label="Seapopym")
lmtl_growth_rate.plot(ax=axis[0], label="LMTL")

axis[0].set_title("Total Mass Conservation (area-weighted)")
axis[0].set_xlabel("Time")
axis[0].set_ylabel("Variation (%)")
axis[0].grid()
axis[0].legend()

# ✅ CORRECTED: Global NRMSE (area-weighted) over time
nrmse_global.plot(ax=axis[1])
axis[1].set_title("Global NRMSE (area-weighted)")
axis[1].grid()
axis[1].set_xlabel("Time")
axis[1].set_ylabel("NRMSE")
axis[1].set_ylim(0, None)

plt.tight_layout()
plt.show()

In [ ]:
# Calculer les moyennes hebdomadaires
# Note: Using spatial fields for visualization (not area-weighted for maps)
rmse_weekly = rmse_spatial  # Already computed as temporal mean
nrmse_weekly = nrmse_spatial  # Already computed as temporal mean
lmtl_weekly = lmtl_biomass.resample(time=interval).mean()
seapopym_weekly = seapopym_biomass.resample(time=interval).mean()

seapopym_weekly = xr.where(seapopym_weekly.notnull(), seapopym_weekly, 0)
lmtl_weekly = xr.where(lmtl_weekly.notnull(), lmtl_weekly, 0)

# Créer la figure pour l'animation
fig = plt.figure(figsize=(18, 12))


def update(frame):
    """Fonction de mise à jour pour chaque frame de l'animation"""
    # Effacer complètement la figure
    fig.clear()

    # Recréer les subplots en grille 2x2
    ax1 = fig.add_subplot(2, 2, 1)
    ax2 = fig.add_subplot(2, 2, 2)
    ax3 = fig.add_subplot(2, 2, 3)
    ax4 = fig.add_subplot(2, 2, 4)

    # RMSE spatial (constant over time - showing temporal average)
    rmse_weekly.plot(cmap="viridis", ax=ax1, vmax=vmax_rmse, vmin=vmin)
    ax1.set_title(f"RMSE (temporal mean)")

    # NRMSE spatial (constant over time - showing temporal average)
    nrmse_weekly.plot(cmap="RdYlGn_r", ax=ax2, vmax=vmax_nrmse, vmin=vmin)
    ax2.set_title(f"NRMSE (temporal mean)")

    # Biomasse LMTL (bas gauche)
    lmtl_weekly.isel(time=frame).plot(cmap="viridis", ax=ax3, vmax=lmtl_weekly.max() / 2)
    ax3.set_title(f"Biomasse LMTL - Week {frame + 1}")

    # Biomasse SeapoPym (bas droite)
    seapopym_weekly.isel(time=frame).plot(cmap="viridis", ax=ax4, vmax=seapopym_weekly.max() / 2)
    ax4.set_title(f"Biomasse SeapoPym - Week {frame + 1}")

    plt.tight_layout()


# Créer l'animation
n_frames = len(lmtl_weekly.time)
anim = FuncAnimation(fig, update, frames=n_frames, interval=500, repeat=True)

# Pour sauvegarder l'animation (décommenter si nécessaire)
_ = anim.save("rmse_nrmse_animation.gif", writer="pillow", fps=10)